# Attention from First Principles

## Learning Objectives

In Module 3, we trained a next-token model using a fixed context window.

For a context of three tokens:

```text
["sat", "on", "the"]
```

we embedded each token and flattened the result:

```text
[B, T, C]
↓
[B, T*C]
```

This worked for a tiny experiment, but it has an important limitation.

The model has no explicit mechanism for asking:

> Which token in the context is relevant to another token?

A Transformer introduces such a mechanism through **self-attention**.

In this notebook we will build attention from basic tensor operations before using any high-level attention implementation.

We will study:

- why fixed context concatenation is limited
- how tokens can interact with other tokens
- queries, keys, and values
- attention scores
- softmax attention weights
- weighted aggregation
- scaled dot-product attention
- why the `1 / sqrt(d_k)` scaling factor exists
- tensor shapes throughout the computation

We will not use:

```python
nn.MultiheadAttention
```

yet.

The goal is to understand what that abstraction eventually performs.

## 1. The Problem We Are Trying to Solve

Suppose a model processes:

```text
the cat sat on the mat
```

Different tokens may need information from different parts of the sequence.

For example, while constructing a representation for:

```text
sat
```

the model may find:

```text
cat
```

more useful than:

```text
the
```

A simple fixed concatenation gives the model all token embeddings, but it does not explicitly calculate relationships between token positions.

Attention introduces a different idea:

```text
for each token:

    compare it with other tokens

    determine how relevant they are

    combine information from them
```

Conceptually:

```text
token
↓
ask what information is relevant
↓
score other tokens
↓
convert scores into weights
↓
weighted combination of their information
```

The weights are learned indirectly through the training objective.

In [2]:
import torch

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1


## 2. Start with a Tiny Sequence

We deliberately begin with only three token representations.

Instead of introducing tokenization, vocabulary construction, and a language-model training loop again, we will initially focus only on the attention mechanism.

Suppose three tokens already have embedding vectors:

```text
token 0
token 1
token 2
```

Each embedding has dimension:

```text
C = 4
```

Therefore the sequence representation has shape:

```text
[T, C]

T = 3
C = 4
```

or:

```text
[3, 4]
```

In [3]:
x = torch.tensor(
    [
        [1.0, 0.0, 1.0, 0.0],
        [0.0, 2.0, 0.0, 2.0],
        [1.0, 1.0, 1.0, 1.0],
    ]
)

print("x:")
print(x)

print("\nShape:")
print(x.shape)

x:
tensor([[1., 0., 1., 0.],
        [0., 2., 0., 2.],
        [1., 1., 1., 1.]])

Shape:
torch.Size([3, 4])


Each row represents one token position:

```text
x[0] → token 0 representation
x[1] → token 1 representation
x[2] → token 2 representation
```

The representation matrix is:

$$
X \in \mathbb{R}^{T \times C}
$$

with:

$$
T = 3
$$

and:

$$
C = 4
$$

So in this example:

$$
X \in \mathbb{R}^{3 \times 4}
$$

The central question is:

> How can each row of `X` construct a new representation using information from the other rows?

## 3. Compare Tokens Using Dot Products

We currently have a sequence representation:

```text
x.shape = [T, C] = [3, 4]
```

where each row is the representation of one token position.

Our next problem is:

> How can one token measure how relevant another token might be?

A simple starting point is the **dot product**.

For two vectors:

$$
a, b \in \mathbb{R}^{C}
$$

their dot product is:

$$
a \cdot b
=
\sum_{i=1}^{C} a_i b_i
$$

The dot product produces a single scalar.

When two vectors point in similar directions, their dot product tends to be larger.

For now, we will use this as a simple measure of alignment between token representations.

Take token `0`.

We can compare it with every token in the sequence:

```text
x[0] · x[0]
x[0] · x[1]
x[0] · x[2]
```

This gives one score for every token position.

Conceptually:

```text
token 0
↓
compare with token 0
compare with token 1
compare with token 2
↓
3 relevance scores
```

Later, queries and keys will make these comparisons learnable.

For now, we deliberately use the token representations directly.

In [4]:
token_0 = x[0]

scores_for_token_0 = x @ token_0

print("Token 0 representation:")
print(token_0)

print("\nScores for token 0:")
print(scores_for_token_0)

print("\nShape:")
print(scores_for_token_0.shape)

Token 0 representation:
tensor([1., 0., 1., 0.])

Scores for token 0:
tensor([2., 0., 2.])

Shape:
torch.Size([3])


## 4. Interpret the Dot Products

Our token representations are:

```text
x[0] = [1, 0, 1, 0]
x[1] = [0, 2, 0, 2]
x[2] = [1, 1, 1, 1]
```

For token `0`, the three dot products are:

### Token 0 with itself

$$
x_0 \cdot x_0
=
(1)(1) + (0)(0) + (1)(1) + (0)(0)
=
2
$$

### Token 0 with token 1

$$
x_0 \cdot x_1
=
(1)(0) + (0)(2) + (1)(0) + (0)(2)
=
0
$$

### Token 0 with token 2

$$
x_0 \cdot x_2
=
(1)(1) + (0)(1) + (1)(1) + (0)(1)
=
2
$$

So token `0` receives the raw scores:

```text
[2, 0, 2]
```

In this deliberately simple example, token `0` is equally aligned with itself and token `2`, while it has zero dot-product alignment with token `1`.

At this stage these are only **raw scores**.

They are not yet probabilities or normalized attention weights.

## 5. Compute Every Pairwise Score at Once

We just computed the comparisons for one token.

But attention needs every token to compare itself with every other token.

Doing that one token at a time would be inefficient.

Matrix multiplication lets us calculate all pairwise dot products simultaneously.

Our representation matrix has shape:

$$
X \in \mathbb{R}^{T \times C}
$$

and its transpose has shape:

$$
X^T \in \mathbb{R}^{C \times T}
$$

Therefore:

$$
S = XX^T
$$

has shape:

$$
[T, C] [C, T]
\rightarrow
[T, T]
$$

For our example:

```text
[3, 4] @ [4, 3]
→
[3, 3]
```

The result is a score matrix:

$$
S \in \mathbb{R}^{T \times T}
$$

where:

$$
S_{ij}
=
x_i \cdot x_j
$$

Each row corresponds to one token asking:

> How strongly do I match every token in the sequence?

Each column corresponds to a token being compared against.

In [5]:
scores = x @ x.T

print("Pairwise score matrix:")
print(scores)

print("\nShape:")
print(scores.shape)

Pairwise score matrix:
tensor([[2., 0., 2.],
        [0., 8., 4.],
        [2., 4., 4.]])

Shape:
torch.Size([3, 3])


## 6. How to Read the Score Matrix

The score matrix has shape:

```text
[T, T] = [3, 3]
```

Suppose it looks like:

```text
[
    [..., ..., ...],
    [..., ..., ...],
    [..., ..., ...]
]
```

Interpret it by row.

The first row:

```text
scores[0]
```

means:

```text
token 0 compared with token 0
token 0 compared with token 1
token 0 compared with token 2
```

The second row:

```text
scores[1]
```

means:

```text
token 1 compared with token 0
token 1 compared with token 1
token 1 compared with token 2
```

and so on.

So we can think of the matrix as:

```text
                 token being compared
              0        1        2
          ┌─────────────────────────
token  0  │
asking 1  │       pairwise scores
       2  │
```

Later, when we introduce queries and keys:

```text
row token    → query
column token → key
```

But we are not introducing those abstractions yet.

First we want to understand the underlying operation.

## 7. Raw Scores Are Not Yet Attention Weights

The pairwise dot products can be any real numbers.

For example:

```text
2
0
2
```

These are useful comparison scores, but they are not yet convenient weights for combining information.

We would like each row to become a set of weights that:

```text
1. are non-negative
2. sum to 1
```

Then we can interpret them as:

> How much attention should this token assign to each token position?

We can obtain such weights using **softmax**.

For a vector of scores:

$$
s_1, s_2, \ldots, s_T
$$

softmax computes:

$$
\alpha_i
=
\frac{e^{s_i}}
{\sum_{j=1}^{T} e^{s_j}}
$$

The resulting values satisfy:

$$
0 < \alpha_i < 1
$$

and:

$$
\sum_i \alpha_i = 1
$$

We apply softmax independently to every row of the score matrix.

In PyTorch:

```python
torch.softmax(scores, dim=-1)
```

`dim=-1` means:

> Apply softmax across the final dimension — across the columns of each row.

So:

```text
raw scores
[T, T]

↓

row-wise softmax

attention weights
[T, T]
```

In [6]:
attention_weights = torch.softmax(
    scores,
    dim=-1,
)

print("Attention weights:")
print(attention_weights)

print("\nShape:")
print(attention_weights.shape)

print("\nRow sums:")
print(attention_weights.sum(dim=-1))

Attention weights:
tensor([[4.6831e-01, 6.3379e-02, 4.6831e-01],
        [3.2932e-04, 9.8169e-01, 1.7980e-02],
        [6.3379e-02, 4.6831e-01, 4.6831e-01]])

Shape:
torch.Size([3, 3])

Row sums:
tensor([1.0000, 1.0000, 1.0000])


## 8. What Does One Attention Row Mean?

Consider:

```python
attention_weights[0]
```

This row contains three numbers:

```text
[
    weight assigned to token 0,
    weight assigned to token 1,
    weight assigned to token 2,
]
```

These weights answer:

> When building a new representation for token 0, how much information should come from each token position?

If the row were:

```text
[0.47, 0.06, 0.47]
```

we could interpret it as:

```text
token 0 receives about 47% of its information from token 0
token 0 receives about  6% from token 1
token 0 receives about 47% from token 2
```

The exact values will come from our actual computation.

### Important distinction

These are **attention weights**, not next-token probabilities.

In Module 3, softmax converted vocabulary logits into:

```text
P(next token | context)
```

Here, softmax is being used for something different:

```text
how strongly one token should combine information
from different token positions
```

Softmax is just a mathematical normalization operation.

Its interpretation depends on what the scores represent.

## 9. Use the Attention Weights to Combine Information

Now every token has a normalized set of weights over all token positions.

The next step is to use those weights to construct a new representation.

For token $i$:

$$
y_i
=
\sum_{j=1}^{T}
\alpha_{ij} x_j
$$

where:

- $x_j$ is the representation of token $j$
- $\alpha_{ij}$ is how much token $i$ attends to token $j$

So each output vector is a weighted combination of the token representations.

For example, if token `0` had weights:

```text
[0.47, 0.06, 0.47]
```

then approximately:

$$
y_0
=
0.47x_0
+
0.06x_1
+
0.47x_2
$$

Instead of calculating this separately for every token, we again use matrix multiplication.

Let:

$$
A \in \mathbb{R}^{T \times T}
$$

be the attention-weight matrix.

Then:

$$
Y = AX
$$

The shapes are:

```text
attention weights     token representations

[T, T]             @  [T, C]

                  ↓

               [T, C]
```

For our example:

```text
[3, 3] @ [3, 4]
→
[3, 4]
```

So attention changes the **contents** of each token representation while preserving the sequence shape:

```text
input:  [T, C]
output: [T, C]
```

In [7]:
attended = attention_weights @ x

print("Original token representations:")
print(x)

print("\nAttention weights:")
print(attention_weights)

print("\nAttended representations:")
print(attended)

print("\nOutput shape:")
print(attended.shape)

Original token representations:
tensor([[1., 0., 1., 0.],
        [0., 2., 0., 2.],
        [1., 1., 1., 1.]])

Attention weights:
tensor([[4.6831e-01, 6.3379e-02, 4.6831e-01],
        [3.2932e-04, 9.8169e-01, 1.7980e-02],
        [6.3379e-02, 4.6831e-01, 4.6831e-01]])

Attended representations:
tensor([[0.9366, 0.5951, 0.9366, 0.5951],
        [0.0183, 1.9814, 0.0183, 1.9814],
        [0.5317, 1.4049, 0.5317, 1.4049]])

Output shape:
torch.Size([3, 4])


## 10. What Just Happened?

We started with three independent token representations:

```text
X
[T, C]
```

Then we performed three operations.

### Step 1 — Compare tokens

```text
X @ X.T
```

produced:

```text
scores
[T, T]
```

Every token received one comparison score for every token position.

### Step 2 — Normalize the scores

```text
softmax(scores)
```

produced:

```text
attention weights
[T, T]
```

Every row now sums to `1`.

### Step 3 — Aggregate information

```text
attention_weights @ X
```

produced:

```text
new token representations
[T, C]
```

Each output token is now a weighted combination of information from the sequence.

The full computation was:

```text
X
[T, C]

↓

X @ X.T

scores
[T, T]

↓

softmax

attention weights
[T, T]

↓

attention_weights @ X

output
[T, C]
```

This is already the core shape of an attention operation:

```text
compare
→ normalize
→ aggregate
```

But there is an important limitation.

We used the same representation `X` for all three purposes:

```text
what a token is looking for
what a token can be matched against
what information a token contributes
```

A Transformer makes these three roles separately learnable.

That is where **queries, keys, and values** come from.

## 11. Interpret the Actual Attention Output

Our pairwise score matrix was:

```text
tensor([
    [2., 0., 2.],
    [0., 8., 4.],
    [2., 4., 4.]
])
```

After row-wise softmax, the attention weights became approximately:

```text
tensor([
    [0.4683, 0.0634, 0.4683],
    [0.0003, 0.9817, 0.0180],
    [0.0634, 0.4683, 0.4683]
])
```

Every row sums to `1`.

### Token 0

Token `0` had raw scores:

```text
[2, 0, 2]
```

so its attention weights became approximately:

```text
[0.4683, 0.0634, 0.4683]
```

It assigns almost equal weight to:

```text
token 0
token 2
```

and much less weight to token `1`.

Its new representation is therefore:

$$
y_0
=
0.4683x_0
+
0.0634x_1
+
0.4683x_2
$$

which produced approximately:

```text
[0.9366, 0.5951, 0.9366, 0.5951]
```

So the output representation for token `0` is no longer simply its original vector.

It now contains information aggregated from the sequence.

---

### Token 1

Token `1` produced the scores:

```text
[0, 8, 4]
```

and therefore attention weights approximately:

```text
[0.0003, 0.9817, 0.0180]
```

Almost all of its attention goes to itself.

Its attended representation therefore stays very close to its original representation:

```text
original:
[0, 2, 0, 2]

attended:
[0.0183, 1.9814, 0.0183, 1.9814]
```

This is not because we explicitly told token `1` that itself was important.

It happened because:

```text
x[1] · x[1] = 8
```

was much larger than its other dot products.

---

## An Important Limitation

Notice that:

```text
x[0] · x[0] = 2
```

while:

```text
x[1] · x[1] = 8
```

Part of this difference comes simply from the magnitudes of the vectors.

The norm of token `1` is larger than the norm of token `0`.

So a raw dot product measures more than just directional similarity:

```text
dot product
=
magnitude interaction
+
directional alignment
```

This is one reason we should not think of these raw scores as a perfect semantic similarity measure.

More importantly, we currently use the same vector `x[i]` for three different jobs:

```text
1. deciding what token i is looking for

2. deciding how token i can be matched

3. deciding what information token i contributes
```

Transformers separate these roles.

That gives us **queries, keys, and values**.

## 12. Why Do We Need Queries, Keys, and Values?

Our first attention mechanism computed:

$$
XX^T
$$

This means the same token representation was used on both sides of every comparison.

Then we also used `X` itself as the information being aggregated:

$$
\text{softmax}(XX^T)X
$$

So one representation had to serve three purposes.

A Transformer instead creates three different representations of every token:

```text
Query  (Q)
Key    (K)
Value  (V)
```

Their roles are different.

### Query

A query represents:

> What information is this token looking for?

### Key

A key represents:

> What kind of information does this token contain or advertise?

### Value

A value represents:

> What information should actually be passed along if this token receives attention?

This gives us the conceptual interaction:

```text
Query
    ↓
compare against Keys
    ↓
attention weights
    ↓
use weights to combine Values
```

Or more compactly:

```text
Q × K
→ determines WHERE to attend

attention weights × V
→ determines WHAT information is retrieved
```

This distinction is fundamental.

The query and key participate in deciding the attention weights.

The value does not determine the match score directly.

It supplies the information that is aggregated after the match has been calculated.

## 13. Learnable Projections

Queries, keys, and values are produced by transforming the original token representations.

Starting from:

$$
X \in \mathbb{R}^{T \times C}
$$

we introduce three parameter matrices:

$$
W_Q
$$

$$
W_K
$$

$$
W_V
$$

and calculate:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

These matrices are trainable parameters in a Transformer.

The training objective eventually determines what useful query, key, and value representations should look like.

They are not manually assigned meanings such as:

```text
query = noun detector
key = subject detector
```

Instead, useful behavior emerges through gradient-based training.

For this first experiment, however, we will use explicit fixed matrices.

That lets us inspect the mechanics without training obscuring what is happening.

## 14. Choose the Projection Dimensions

Our original token representation has dimension:

```text
C = 4
```

For this experiment, we will choose:

```text
d_k = 3
d_v = 2
```

where:

- `d_k` is the query/key dimension
- `d_v` is the value dimension

Queries and keys must have compatible dimensions because we will take their dot products.

Therefore:

```text
Q shape = [T, d_k]
K shape = [T, d_k]
```

Values do not need the same final dimension:

```text
V shape = [T, d_v]
```

The projection matrices therefore have shapes:

```text
W_Q : [C, d_k] = [4, 3]
W_K : [C, d_k] = [4, 3]
W_V : [C, d_v] = [4, 2]
```

and:

```text
X [3, 4] @ W_Q [4, 3]
→ Q [3, 3]

X [3, 4] @ W_K [4, 3]
→ K [3, 3]

X [3, 4] @ W_V [4, 2]
→ V [3, 2]
```

Using different `d_k` and `d_v` here makes an important fact visible:

> The vectors used to decide attention do not have to be identical to the information vectors that are ultimately aggregated.

In [8]:
d_k = 3
d_v = 2

W_Q = torch.tensor(
    [
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.0, 0.0, 1.0],
        [1.0, 1.0, 0.0],
    ]
)

W_K = torch.tensor(
    [
        [1.0, 0.0, 1.0],
        [0.0, 1.0, 0.0],
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 1.0],
    ]
)

W_V = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
        [0.0, 1.0],
    ]
)

print("W_Q shape:", W_Q.shape)
print("W_K shape:", W_K.shape)
print("W_V shape:", W_V.shape)

W_Q shape: torch.Size([4, 3])
W_K shape: torch.Size([4, 3])
W_V shape: torch.Size([4, 2])


## 15. Project the Tokens into Q, K, and V

Now we calculate:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

Notice that all three originate from the same original token representations:

```text
             ┌→ Q
X ───────────┼→ K
             └→ V
```

but each projection can represent a different aspect of the token.

This is much more flexible than forcing the original embedding `X` to perform every role directly.

In [9]:
Q = x @ W_Q
K = x @ W_K
V = x @ W_V

print("Q:")
print(Q)
print("shape:", Q.shape)

print("\nK:")
print(K)
print("shape:", K.shape)

print("\nV:")
print(V)
print("shape:", V.shape)

Q:
tensor([[1., 0., 1.],
        [2., 4., 0.],
        [2., 2., 1.]])
shape: torch.Size([3, 3])

K:
tensor([[2., 0., 1.],
        [0., 4., 2.],
        [2., 2., 2.]])
shape: torch.Size([3, 3])

V:
tensor([[2., 1.],
        [0., 4.],
        [2., 3.]])
shape: torch.Size([3, 2])


## 16. Query-Key Scores

Now we determine which tokens should attend to which other tokens.

Instead of:

$$
XX^T
$$

we calculate:

$$
QK^T
$$

The shapes are:

```text
Q       K.T

[T,d_k] @ [d_k,T]

        ↓

      [T,T]
```

For our example:

```text
[3, 3] @ [3, 3]
→
[3, 3]
```

The entry:

$$
(QK^T)_{ij}
$$

is the dot product between:

```text
query of token i
```

and:

```text
key of token j
```

So row `i` asks:

> How strongly does the query from token `i` match each token's key?

This is the origin of the names **query** and **key**.

A query is compared against keys to determine relevance.

In [10]:
qk_scores = Q @ K.T

print("Query-key score matrix:")
print(qk_scores)

print("\nShape:")
print(qk_scores.shape)

Query-key score matrix:
tensor([[ 3.,  2.,  4.],
        [ 4., 16., 12.],
        [ 5., 10., 10.]])

Shape:
torch.Size([3, 3])


## 17. Normalize the Query-Key Scores

As before, the raw scores are not yet normalized attention weights.

For now we apply row-wise softmax directly:

$$
A
=
\text{softmax}(QK^T)
$$

Later in this notebook we will modify this equation by introducing the important scaling factor:

$$
\frac{1}{\sqrt{d_k}}
$$

For the moment, we deliberately leave scaling out so we can isolate the role of Q, K, and V.

In [11]:
qkv_attention_weights = torch.softmax(
    qk_scores,
    dim=-1,
)

print("QKV attention weights:")
print(qkv_attention_weights)

print("\nRow sums:")
print(
    qkv_attention_weights.sum(
        dim=-1
    )
)

QKV attention weights:
tensor([[2.4473e-01, 9.0031e-02, 6.6524e-01],
        [6.0337e-06, 9.8201e-01, 1.7986e-02],
        [3.3577e-03, 4.9832e-01, 4.9832e-01]])

Row sums:
tensor([1.0000, 1.0000, 1.0000])


## 18. Aggregate Values, Not Keys

Now comes the distinction that makes Q, K, and V easier to remember.

Queries and keys determined the attention weights:

```text
Q @ K.T
↓
softmax
↓
attention weights
```

But the thing we aggregate is **V**:

$$
Y = AV
$$

where:

```text
A = attention weights [T, T]
V = values            [T, d_v]
```

Therefore:

```text
[T, T] @ [T, d_v]
→
[T, d_v]
```

In our example:

```text
[3, 3] @ [3, 2]
→
[3, 2]
```

The sequence still contains three token positions, but each resulting token representation now has dimension `d_v = 2`.

This gives the complete unscaled QKV attention operation:

```text
X
│
├── X @ W_Q → Q
├── X @ W_K → K
└── X @ W_V → V

Q @ K.T
↓
scores
↓
softmax
↓
attention weights
↓
attention weights @ V
↓
output
```

In [12]:
qkv_output = (
    qkv_attention_weights
    @ V
)

print("Values:")
print(V)

print("\nAttention weights:")
print(qkv_attention_weights)

print("\nQKV attention output:")
print(qkv_output)

print("\nOutput shape:")
print(qkv_output.shape)

Values:
tensor([[2., 1.],
        [0., 4.],
        [2., 3.]])

Attention weights:
tensor([[2.4473e-01, 9.0031e-02, 6.6524e-01],
        [6.0337e-06, 9.8201e-01, 1.7986e-02],
        [3.3577e-03, 4.9832e-01, 4.9832e-01]])

QKV attention output:
tensor([[1.8199, 2.6006],
        [0.0360, 3.9820],
        [1.0034, 3.4916]])

Output shape:
torch.Size([3, 2])


## 19. The Core QKV Equation So Far

We have now built the central attention computation:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

then:

$$
A
=
\text{softmax}(QK^T)
$$

and finally:

$$
Y = AV
$$

Combining those pieces:

$$
Y
=
\text{softmax}(QK^T)V
$$

This is almost scaled dot-product attention.

The actual Transformer version adds one crucial factor:

$$
\text{Attention}(Q,K,V)
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

We have not yet justified the division by:

$$
\sqrt{d_k}
$$

Rather than accepting it as part of a formula, we will next construct an experiment showing **why unscaled dot products become problematic as the query/key dimension grows**.

## 20. Interpret the Actual Q, K, and V Results

Our projections produced:

```text
Q =
[[1, 0, 1],
 [2, 4, 0],
 [2, 2, 1]]

K =
[[2, 0, 1],
 [0, 4, 2],
 [2, 2, 2]]

V =
[[2, 1],
 [0, 4],
 [2, 3]]
```

Although all three were derived from the same original matrix `X`, they are now different representations.

They perform different jobs.

```text
Q
→ what each token uses to search

K
→ what each token uses to be matched

V
→ what information each token contributes
```

---

### Query-key interactions

The query-key score matrix was:

```text
QKᵀ =

[[ 3,  2,  4],
 [ 4, 16, 12],
 [ 5, 10, 10]]
```

For token `0`, the query produced scores:

```text
[3, 2, 4]
```

After softmax:

```text
[0.2447, 0.0900, 0.6652]
```

So token `0` now attends most strongly to token `2`.

This differs from our earlier experiment using:

```text
X @ X.T
```

where token `0` gave approximately equal attention to token `0` and token `2`.

The difference occurred because the model is no longer comparing the original token representations directly.

Instead it compares:

```text
query representations
against
key representations
```

This is the main purpose of the learned Q/K projections:

> The model can learn a representation specifically for deciding which tokens are relevant to each other.

---

### Token 1

Token `1` produced scores:

```text
[4, 16, 12]
```

which softmax converted to approximately:

```text
[0.000006, 0.9820, 0.0180]
```

This distribution is extremely concentrated on token `1`.

The large difference between the logits:

```text
16 vs 12 vs 4
```

becomes even more pronounced after exponentiation inside softmax.

This observation will become important when we study attention scaling.

---

### Token 2

Token `2` produced:

```text
[5, 10, 10]
```

and therefore:

```text
[0.0034, 0.4983, 0.4983]
```

Token `2` assigns almost exactly equal attention to tokens `1` and `2`.

Again, attention preserves uncertainty when the relevant scores are equal.

---

## Values Determine What Is Retrieved

The attention weights were then applied to `V`, not to `K` or `Q`.

For token `0`:

```text
attention weights:
[0.2447, 0.0900, 0.6652]
```

and:

```text
V =
token 0 → [2, 1]
token 1 → [0, 4]
token 2 → [2, 3]
```

Therefore the output is:

$$
y_0
=
0.2447v_0
+
0.0900v_1
+
0.6652v_2
$$

which produced approximately:

```text
[1.8199, 2.6006]
```

This reinforces the distinction:

```text
Q and K
→ determine attention weights

V
→ supplies the information being combined
```

---

## Shape Transformation

The original sequence had shape:

```text
X: [T, C] = [3, 4]
```

Our values had dimension:

```text
d_v = 2
```

so the final attention output has shape:

```text
[T, d_v] = [3, 2]
```

Attention therefore preserves the number of token positions:

```text
T = 3
```

while the feature dimension of the output is determined by the value representation.

Later, Transformer architectures will use additional projections so these dimensions fit cleanly into residual blocks.

## 21. Why Might Raw Dot Products Become a Problem?

Our attention formula so far is:

$$
\text{softmax}(QK^T)V
$$

But the Transformer uses:

$$
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

Why divide by:

$$
\sqrt{d_k}
$$

?

To answer this, consider a dot product:

$$
q \cdot k
=
\sum_{i=1}^{d_k} q_i k_i
$$

As `d_k` grows, we are summing more terms.

Even if the individual components of `q` and `k` have roughly similar statistical scales, the magnitude and variance of their dot products tend to grow with the number of dimensions.

That means larger `d_k` can produce increasingly large positive and negative attention scores.

Why could that be a problem?

Because those scores are passed into:

```text
softmax
```

and softmax contains exponentials.

Large score differences can cause softmax to become extremely concentrated:

```text
one token ≈ 1
other tokens ≈ 0
```

before the model has necessarily learned that such extreme confidence is useful.

We will test this directly.

## 22. Measure Dot-Product Scale as Dimension Grows

We will generate random query and key vectors whose components have approximately:

```text
mean = 0
variance = 1
```

Then we will measure the dot products for different dimensions:

```text
d_k = 4
d_k = 16
d_k = 64
d_k = 256
d_k = 1024
```

If each product term has roughly constant variance, summing `d_k` such terms should make the variance of the total dot product grow roughly in proportion to:

$$
d_k
$$

Therefore its standard deviation should grow approximately like:

$$
\sqrt{d_k}
$$

This is the statistical reason behind the Transformer scaling factor.

In [13]:
torch.manual_seed(42)

dimensions = [
    4,
    16,
    64,
    256,
    1024,
]

num_samples = 10_000

for d_k_test in dimensions:
    q = torch.randn(
        num_samples,
        d_k_test,
    )

    k = torch.randn(
        num_samples,
        d_k_test,
    )

    dot_products = (
        q * k
    ).sum(dim=-1)

    print(
        f"d_k={d_k_test:4d} | "
        f"std(dot product)="
        f"{dot_products.std().item():.4f} | "
        f"sqrt(d_k)="
        f"{d_k_test ** 0.5:.4f}"
    )

d_k=   4 | std(dot product)=2.0382 | sqrt(d_k)=2.0000
d_k=  16 | std(dot product)=4.0151 | sqrt(d_k)=4.0000
d_k=  64 | std(dot product)=7.9707 | sqrt(d_k)=8.0000
d_k= 256 | std(dot product)=16.0130 | sqrt(d_k)=16.0000
d_k=1024 | std(dot product)=31.7158 | sqrt(d_k)=32.0000


## 23. What We Expect

If the components of `q` and `k` are independent and approximately standard normal:

$$
q_i, k_i \sim \mathcal{N}(0,1)
$$

then each product:

$$
q_i k_i
$$

has approximately:

$$
\mathbb{E}[q_i k_i] = 0
$$

and variance near:

$$
1
$$

The dot product is:

$$
q \cdot k
=
\sum_{i=1}^{d_k} q_i k_i
$$

For approximately independent terms, variances add:

$$
\operatorname{Var}(q \cdot k)
\approx d_k
$$

Therefore:

$$
\operatorname{Std}(q \cdot k)
\approx \sqrt{d_k}
$$

So if we divide the dot product by:

$$
\sqrt{d_k}
$$

the standard deviation should return to approximately:

$$
1
$$

regardless of the query/key dimension.

Let's test that next.

In [15]:
torch.manual_seed(42)

for d_k_test in dimensions:
    q = torch.randn(
        num_samples,
        d_k_test,
    )

    k = torch.randn(
        num_samples,
        d_k_test,
    )

    dot_products = (
        q * k
    ).sum(dim=-1)

    scaled_dot_products = (
        dot_products
        / (d_k_test ** 0.5)
    )

    print(
        f"d_k={d_k_test:4d} | "
        f"unscaled std="
        f"{dot_products.std().item():.4f} | "
        f"scaled std="
        f"{scaled_dot_products.std().item():.4f}"
    )

d_k=   4 | unscaled std=2.0382 | scaled std=1.0191
d_k=  16 | unscaled std=4.0151 | scaled std=1.0038
d_k=  64 | unscaled std=7.9707 | scaled std=0.9963
d_k= 256 | unscaled std=16.0130 | scaled std=1.0008
d_k=1024 | unscaled std=31.7158 | scaled std=0.9911


## 24. Why Large Dot Products Can Make Softmax Too Sharp

We just observed:

```text
Std(q · k) ≈ sqrt(d_k)
```

and:

```text
Std((q · k) / sqrt(d_k)) ≈ 1
```

Now we need to connect that to softmax.

Suppose one token produces attention scores such as:

```text
[1.0, 1.5, 2.0]
```

Softmax will produce a moderately distributed set of weights.

But if the same relative pattern is scaled up:

```text
[10, 15, 20]
```

softmax becomes much more concentrated on the largest score.

This happens because softmax exponentiates its inputs:

$$
\text{softmax}(z_i)
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}
$$

Larger score differences therefore produce exponentially larger probability ratios.

For attention, this means:

```text
large dot-product variance
↓
large score differences
↓
very sharp softmax
↓
one or a few positions dominate
```

If this happens simply because `d_k` is large, rather than because the model learned a meaningful preference, optimization becomes harder.

Let's demonstrate that directly.

In [16]:
example_scores = torch.tensor(
    [1.0, 1.5, 2.0]
)

scales = [
    1.0,
    2.0,
    5.0,
    10.0,
]

for scale in scales:
    scaled_scores = (
        example_scores * scale
    )

    weights = torch.softmax(
        scaled_scores,
        dim=-1,
    )

    print(
        f"scale={scale:>4}: "
        f"scores={scaled_scores.tolist()} "
        f"softmax={weights.tolist()}"
    )

scale= 1.0: scores=[1.0, 1.5, 2.0] softmax=[0.18632373213768005, 0.30719590187072754, 0.5064803957939148]
scale= 2.0: scores=[2.0, 3.0, 4.0] softmax=[0.09003057330846786, 0.2447284758090973, 0.6652409434318542]
scale= 5.0: scores=[5.0, 7.5, 10.0] softmax=[0.006188285071402788, 0.07538874447345734, 0.9184229373931885]
scale=10.0: scores=[10.0, 15.0, 20.0] softmax=[4.509404243435711e-05, 0.006692549213767052, 0.9932623505592346]


## 25. Softmax Sharpness Depends on Score Scale

The ranking of the scores does not change when we multiply them by a positive constant.

For example:

```text
1.0 < 1.5 < 2.0
```

and:

```text
10 < 15 < 20
```

have the same ordering.

But softmax does not only care about ordering.

It also cares about the **distance between scores**.

As those distances grow:

```text
softmax becomes sharper
```

and the largest score receives increasingly dominant weight.

This is directly analogous to the temperature experiment from Module 3.

There we used:

$$
\text{softmax}(z / T)
$$

Lower temperature magnified score differences.

Here, increasing `d_k` can unintentionally increase dot-product scale.

So without correction, larger attention dimensions behave somewhat like an unintended reduction in temperature.

The Transformer scaling factor counteracts that effect.

## 26. Measure Attention Sharpness as d_k Grows

Now we will simulate a more realistic attention setup.

For each query/key dimension:

```text
d_k = 4
16
64
256
1024
```

we will generate random queries and keys for a small sequence.

For each dimension we will compute two attention distributions:

```text
unscaled:
softmax(QKᵀ)

scaled:
softmax(QKᵀ / sqrt(d_k))
```

We need a quantitative way to measure how sharp the distributions are.

One useful measure is the **maximum attention weight** in each row.

If attention is relatively spread out:

```text
max weight might be around 0.2 or 0.3
```

If it becomes nearly one-hot:

```text
max weight approaches 1
```

We will average that maximum over many random examples.

In [17]:
torch.manual_seed(42)

sequence_length = 16
num_trials = 1_000

for d_k_test in dimensions:
    unscaled_max_weights = []
    scaled_max_weights = []

    for _ in range(num_trials):
        q = torch.randn(
            sequence_length,
            d_k_test,
        )

        k = torch.randn(
            sequence_length,
            d_k_test,
        )

        scores = q @ k.T

        unscaled_weights = torch.softmax(
            scores,
            dim=-1,
        )

        scaled_weights = torch.softmax(
            scores / (d_k_test ** 0.5),
            dim=-1,
        )

        unscaled_max_weights.append(
            unscaled_weights.max(
                dim=-1
            ).values.mean()
        )

        scaled_max_weights.append(
            scaled_weights.max(
                dim=-1
            ).values.mean()
        )

    unscaled_mean = torch.stack(
        unscaled_max_weights
    ).mean()

    scaled_mean = torch.stack(
        scaled_max_weights
    ).mean()

    print(
        f"d_k={d_k_test:4d} | "
        f"unscaled avg max="
        f"{unscaled_mean.item():.4f} | "
        f"scaled avg max="
        f"{scaled_mean.item():.4f}"
    )

d_k=   4 | unscaled avg max=0.4315 | scaled avg max=0.2376
d_k=  16 | unscaled avg max=0.6849 | scaled avg max=0.2434
d_k=  64 | unscaled avg max=0.8433 | scaled avg max=0.2449
d_k= 256 | unscaled avg max=0.9231 | scaled avg max=0.2471
d_k=1024 | unscaled avg max=0.9606 | scaled avg max=0.2463


## 27. What We Expect from the Sharpness Experiment

For random queries and keys:

```text
unscaled QKᵀ
```

should become increasingly extreme as `d_k` grows.

Therefore we expect:

```text
unscaled average max attention weight
↑ as d_k grows
```

In contrast:

```text
QKᵀ / sqrt(d_k)
```

keeps the score variance approximately stable.

So we expect:

```text
scaled average max attention weight
≈ relatively stable across d_k
```

This does not mean attention should always be diffuse.

A trained model may learn very sharp attention when that is useful.

The point is different:

> The sharpness should come from learned structure, not simply from increasing the dimensionality of Q and K.

## 28. Entropy Gives Another View of Attention Sharpness

Maximum probability tells us how dominant the strongest position is.

Another useful measure is **entropy**.

For an attention distribution:

$$
p_1, p_2, \ldots, p_T
$$

entropy is:

$$
H(p)
=
-\sum_i p_i \log p_i
$$

A high-entropy distribution is relatively spread out.

A low-entropy distribution is concentrated on a small number of positions.

Examples:

```text
uniform attention
→ high entropy

nearly one-hot attention
→ low entropy
```

If unscaled dot products become excessively large as `d_k` increases, we should see their attention entropy decrease.

Scaled attention should remain much more stable.

In [18]:
torch.manual_seed(42)

for d_k_test in dimensions:
    unscaled_entropies = []
    scaled_entropies = []

    for _ in range(num_trials):
        q = torch.randn(
            sequence_length,
            d_k_test,
        )

        k = torch.randn(
            sequence_length,
            d_k_test,
        )

        scores = q @ k.T

        unscaled_weights = torch.softmax(
            scores,
            dim=-1,
        )

        scaled_weights = torch.softmax(
            scores / (d_k_test ** 0.5),
            dim=-1,
        )

        unscaled_entropy = -(
            unscaled_weights
            * torch.log(
                unscaled_weights + 1e-12
            )
        ).sum(dim=-1).mean()

        scaled_entropy = -(
            scaled_weights
            * torch.log(
                scaled_weights + 1e-12
            )
        ).sum(dim=-1).mean()

        unscaled_entropies.append(
            unscaled_entropy
        )

        scaled_entropies.append(
            scaled_entropy
        )

    unscaled_mean_entropy = (
        torch.stack(
            unscaled_entropies
        ).mean()
    )

    scaled_mean_entropy = (
        torch.stack(
            scaled_entropies
        ).mean()
    )

    print(
        f"d_k={d_k_test:4d} | "
        f"unscaled entropy="
        f"{unscaled_mean_entropy.item():.4f} | "
        f"scaled entropy="
        f"{scaled_mean_entropy.item():.4f}"
    )

d_k=   4 | unscaled entropy=1.7523 | scaled entropy=2.3753
d_k=  16 | unscaled entropy=0.9034 | scaled entropy=2.3632
d_k=  64 | unscaled entropy=0.4117 | scaled entropy=2.3612
d_k= 256 | unscaled entropy=0.1918 | scaled entropy=2.3564
d_k=1024 | unscaled entropy=0.0956 | scaled entropy=2.3578


## 29. Scaled Dot-Product Attention

We can now motivate the Transformer attention equation from the experiments rather than memorizing it.

We start with learned projections:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

The raw query-key interaction is:

$$
QK^T
$$

But its variance grows approximately with:

$$
d_k
$$

and its standard deviation grows approximately with:

$$
\sqrt{d_k}
$$

So we normalize the scores:

$$
\frac{QK^T}{\sqrt{d_k}}
$$

Then apply row-wise softmax:

$$
A
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
$$

Finally we aggregate values:

$$
Y = AV
$$

Combining everything gives:

$$
\boxed{
\text{Attention}(Q,K,V)
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

This is **scaled dot-product attention**.

The scaling term does not change which key has the largest raw score.

It changes the scale of the scores before softmax so that attention behavior remains numerically better behaved as `d_k` grows.

## 30. Apply Scaling to Our Original QKV Example

Earlier our query-key score matrix was:

```text
QKᵀ =

[[ 3,  2,  4],
 [ 4, 16, 12],
 [ 5, 10, 10]]
```

Our query/key dimension is:

```text
d_k = 3
```

So scaled dot-product attention uses:

$$
\frac{QK^T}{\sqrt{3}}
$$

We will compare:

```text
unscaled attention weights
```

against:

```text
scaled attention weights
```

using the same Q, K, and V.

This lets us see exactly what scaling changes in the example we already understand.

In [19]:
scale = d_k ** 0.5

scaled_qk_scores = (
    qk_scores / scale
)

scaled_attention_weights = torch.softmax(
    scaled_qk_scores,
    dim=-1,
)

scaled_qkv_output = (
    scaled_attention_weights
    @ V
)

print(
    "sqrt(d_k):",
    scale,
)

print(
    "\nOriginal QK scores:"
)
print(qk_scores)

print(
    "\nScaled QK scores:"
)
print(scaled_qk_scores)

print(
    "\nUnscaled attention weights:"
)
print(qkv_attention_weights)

print(
    "\nScaled attention weights:"
)
print(scaled_attention_weights)

print(
    "\nScaled attention output:"
)
print(scaled_qkv_output)

sqrt(d_k): 1.7320508075688772

Original QK scores:
tensor([[ 3.,  2.,  4.],
        [ 4., 16., 12.],
        [ 5., 10., 10.]])

Scaled QK scores:
tensor([[1.7321, 1.1547, 2.3094],
        [2.3094, 9.2376, 6.9282],
        [2.8868, 5.7735, 5.7735]])

Unscaled attention weights:
tensor([[2.4473e-01, 9.0031e-02, 6.6524e-01],
        [6.0337e-06, 9.8201e-01, 1.7986e-02],
        [3.3577e-03, 4.9832e-01, 4.9832e-01]])

Scaled attention weights:
tensor([[2.9916e-01, 1.6794e-01, 5.3290e-01],
        [8.9045e-04, 9.0884e-01, 9.0267e-02],
        [2.7122e-02, 4.8644e-01, 4.8644e-01]])

Scaled attention output:
tensor([[1.6641, 2.5696],
        [0.1823, 3.9071],
        [1.0271, 3.4322]])


## 31. What Scaling Does — and Does Not Do

Scaling by:

$$
\sqrt{d_k}
$$

does not change the ordering of the query-key scores.

For any positive constant `c`:

```text
a > b
```

implies:

```text
a / c > b / c
```

So the same token remains the highest-scoring key.

What changes is the **distance between scores before softmax**.

That changes how concentrated the resulting attention distribution becomes.

Therefore scaling affects:

```text
confidence / sharpness
```

but not:

```text
raw score ranking
```

This is closely related to temperature scaling:

$$
\text{softmax}(z/T)
$$

except here the scale:

$$
T = \sqrt{d_k}
$$

is determined by the query/key dimensionality rather than chosen as a decoding hyperparameter.

## 32. Interpret the Scaling Experiments

The experiments give us three complementary views of why scaled dot-product attention uses:

$$
\frac{QK^T}{\sqrt{d_k}}
$$

---

### 1. Softmax becomes sharper when score magnitudes grow

Starting with:

```text
[1.0, 1.5, 2.0]
```

we multiplied the entire score vector by increasingly large constants.

The maximum softmax probability changed approximately as:

```text
scale = 1   → 0.506
scale = 2   → 0.665
scale = 5   → 0.918
scale = 10  → 0.993
```

The ordering of the scores never changed.

Only their separation changed.

Yet softmax moved from a distributed probability vector toward an almost one-hot vector.

This demonstrates:

```text
larger score differences
→ sharper softmax
```

---

### 2. Increasing d_k causes unscaled attention to become artificially sharp

For random queries and keys, we measured the average largest attention weight:

```text
d_k       unscaled     scaled

4         0.4315       0.2376
16        0.6849       0.2434
64        0.8433       0.2449
256       0.9231       0.2471
1024      0.9606       0.2463
```

Without scaling, merely increasing the query/key dimension caused the largest attention weight to approach:

```text
1
```

At:

```text
d_k = 1024
```

a random attention row already assigns, on average, about:

```text
96%
```

of its mass to a single position.

This extreme concentration is caused by dimensionality, not by learned structure.

With scaling, the average maximum weight remained around:

```text
0.24 – 0.25
```

across every tested dimension.

---

### 3. Entropy confirms the same effect

We also measured attention entropy.

For a distribution:

$$
p_1,\ldots,p_T
$$

entropy is:

$$
H(p)
=
-\sum_i p_i \log p_i
$$

Our results were:

```text
d_k       unscaled     scaled

4         1.7523       2.3753
16        0.9034       2.3632
64        0.4117       2.3612
256       0.1918       2.3564
1024      0.0956       2.3578
```

The unscaled entropy collapses toward zero as dimensionality increases.

That means attention becomes increasingly concentrated.

The scaled entropy remains approximately constant.

For reference, a perfectly uniform distribution across 16 positions has entropy:

$$
\log(16) \approx 2.773
$$

So scaled attention is not forced to be uniform.

It simply avoids becoming artificially sharp because `d_k` increased.

---

## Scaling Our Original Attention Example

Our original query-key scores were:

```text
[[ 3,  2,  4],
 [ 4, 16, 12],
 [ 5, 10, 10]]
```

Because:

```text
d_k = 3
```

we divide by:

$$
\sqrt{3} \approx 1.732
$$

For token `1`, the unscaled attention weights were approximately:

```text
[0.000006, 0.9820, 0.0180]
```

After scaling:

```text
[0.000890, 0.9088, 0.0903]
```

Token `1` still prefers the same position.

Scaling did not change the ranking.

Instead, it made the distribution less excessively confident.

This is the key role of the scaling factor:

> Keep the scale of attention logits statistically controlled so that softmax sharpness does not grow merely because the query/key dimension is larger.

The model remains free to learn sharp attention when the data supports it.

# Causal Self-Attention

## 33. A Problem for Autoregressive Language Modeling

So far, our attention matrix allows every token to attend to every position.

For a sequence:

```text
the cat sat on the mat
```

a token at position `1` could theoretically attend to:

```text
position 0
position 1
position 2
position 3
position 4
position 5
```

including tokens that occur **after** it.

That is acceptable for some kinds of models.

But it is not acceptable for an autoregressive next-token language model.

Recall the objective:

$$
P(x_t \mid x_{<t})
$$

When predicting token `t`, the model must only use tokens that occurred before it.

It must not have access to future tokens.

Otherwise training would leak the answer.

## 34. Why Future Attention Would Be Cheating

Suppose we train on:

```text
the cat sat on the mat
```

and ask the model to predict:

```text
sat
```

from:

```text
the cat
```

If the representation for `"cat"` were allowed to attend to future positions, it could inspect:

```text
sat
on
the
mat
```

during training.

The model would therefore receive information that will not exist during real autoregressive generation.

Training would no longer match the generation problem.

We need an attention rule:

```text
token at position i
may attend only to positions j <= i
```

This is called **causal attention**.

It is implemented using a **causal mask**.

## 35. Visualize the Allowed Attention Pattern

For a sequence with four positions:

```text
0  1  2  3
```

causal attention should permit:

```text
position 0 → 0

position 1 → 0, 1

position 2 → 0, 1, 2

position 3 → 0, 1, 2, 3
```

As a matrix:

```text
             key position
            0  1  2  3

query  0    ✓  ✗  ✗  ✗
       1    ✓  ✓  ✗  ✗
       2    ✓  ✓  ✓  ✗
       3    ✓  ✓  ✓  ✓
```

This is a lower-triangular pattern.

PyTorch can create it directly with:

```python
torch.tril(...)
```

In [20]:
T = 4

causal_mask = torch.tril(
    torch.ones(T, T)
)

print("Causal mask:")
print(causal_mask)

Causal mask:
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


## 36. Why We Cannot Simply Multiply Scores by the Mask

A tempting implementation would be:

```python
masked_scores = scores * causal_mask
```

But this is wrong.

Suppose a forbidden future position has score:

```text
5
```

Multiplying by zero gives:

```text
0
```

Then softmax still assigns:

$$
e^0 = 1
$$

to that position.

So it would still receive nonzero attention.

We need forbidden positions to receive effectively **zero probability after softmax**.

To achieve that, we replace their scores with:

$$
-\infty
$$

because:

$$
e^{-\infty} = 0
$$

Then softmax assigns exactly zero probability to those positions.

In [21]:
example_scores = torch.tensor(
    [
        [2.0, 1.0, 3.0, 4.0],
        [1.0, 3.0, 2.0, 4.0],
        [2.0, 1.0, 4.0, 3.0],
        [1.0, 2.0, 3.0, 4.0],
    ]
)

print("Original scores:")
print(example_scores)

print("\nCausal mask:")
print(causal_mask)

Original scores:
tensor([[2., 1., 3., 4.],
        [1., 3., 2., 4.],
        [2., 1., 4., 3.],
        [1., 2., 3., 4.]])

Causal mask:
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [22]:
masked_scores = example_scores.masked_fill(
    causal_mask == 0,
    float("-inf"),
)

print("Masked scores:")
print(masked_scores)

Masked scores:
tensor([[2., -inf, -inf, -inf],
        [1., 3., -inf, -inf],
        [2., 1., 4., -inf],
        [1., 2., 3., 4.]])


## 37. Apply Softmax After Masking

Now we apply softmax row by row.

The `-inf` entries contribute:

$$
e^{-\infty}=0
$$

so forbidden future positions receive exactly zero attention probability.

The remaining allowed positions are renormalized so that each row still sums to:

$$
1
$$

In [23]:
causal_attention_weights = torch.softmax(
    masked_scores,
    dim=-1,
)

print("Causal attention weights:")
print(causal_attention_weights)

print("\nRow sums:")
print(
    causal_attention_weights.sum(
        dim=-1
    )
)

Causal attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.1192, 0.8808, 0.0000, 0.0000],
        [0.1142, 0.0420, 0.8438, 0.0000],
        [0.0321, 0.0871, 0.2369, 0.6439]])

Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000])


## 38. Interpret the Causal Attention Weights

The causal mask enforces:

```text
position i
may attend only to
positions j <= i
```

Our resulting attention weights were:

```text
[
    [1.0000, 0.0000, 0.0000, 0.0000],
    [0.1192, 0.8808, 0.0000, 0.0000],
    [0.1142, 0.0420, 0.8438, 0.0000],
    [0.0321, 0.0871, 0.2369, 0.6439],
]
```

### Position 0

Position `0` has no earlier tokens.

So it can attend only to itself:

```text
[1, 0, 0, 0]
```

### Position 1

Position `1` may attend to:

```text
position 0
position 1
```

but not:

```text
position 2
position 3
```

Its allowed scores were:

```text
[1, 3]
```

so after softmax it assigns approximately:

```text
11.9% → position 0
88.1% → position 1
```

### Position 2

Position `2` may attend to:

```text
0, 1, 2
```

but not position `3`.

### Position 3

Position `3` is the final position, so there are no future tokens to hide.

It may attend to all four positions.

---

The important result is:

```text
future attention probability = 0
```

while:

```text
allowed probabilities still sum to 1
```

This lets every token build a representation using only information available up to that point in the sequence.

## 39. Why This Matters for Next-Token Training

A decoder-only language model can process an entire sequence in parallel during training.

Suppose the training sequence is:

```text
the cat sat on
```

and the next-token targets are:

```text
cat sat on the
```

We want the model to learn:

```text
the
→ predict cat

the cat
→ predict sat

the cat sat
→ predict on

the cat sat on
→ predict the
```

All four prediction positions can be computed in one forward pass.

But that is only valid if each position is prevented from seeing future tokens.

The causal mask ensures exactly that.

Conceptually:

```text
input positions:

the   cat   sat   on
 0     1     2     3

position 0 sees:
the

position 1 sees:
the cat

position 2 sees:
the cat sat

position 3 sees:
the cat sat on
```

So although training is parallelized across positions, the information available to each position still respects the autoregressive constraint.

This is a crucial idea:

> Causal masking lets us train many next-token predictions in parallel without leaking future information.

## 40. Put Scaling and Causal Masking Together

We now have all the pieces needed for decoder-style attention.

Starting from token representations:

$$
X
$$

we compute:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

Then compute scaled query-key scores:

$$
S
=
\frac{QK^T}{\sqrt{d_k}}
$$

Next, causal masking modifies forbidden future entries:

$$
S_{ij}
=
-\infty
\quad \text{when } j > i
$$

Then:

$$
A
=
\text{softmax}(S)
$$

Finally:

$$
Y = AV
$$

So the complete causal scaled dot-product attention operation is:

$$
\boxed{
\text{CausalAttention}(Q,K,V)
=
\text{softmax}
\left(
\text{mask}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
\right)V
}
$$

This is the core attention operation used inside a decoder-only Transformer.

## 41. Build Causal Scaled Dot-Product Attention as a Function

So far we have implemented every step separately.

Now we will combine them into one reusable function.

The function will receive:

```text
Q: [T, d_k]
K: [T, d_k]
V: [T, d_v]
```

and return:

```text
output:            [T, d_v]
attention weights: [T, T]
```

The computation will be:

```text
Q @ K.T
↓
divide by sqrt(d_k)
↓
apply causal mask
↓
softmax
↓
multiply by V
```

In [24]:
def causal_scaled_dot_product_attention(
    Q,
    K,
    V,
):
    d_k_local = Q.shape[-1]

    scores = (
        Q @ K.T
        / (d_k_local ** 0.5)
    )

    T_local = Q.shape[0]

    mask = torch.tril(
        torch.ones(
            T_local,
            T_local,
            dtype=torch.bool,
        )
    )

    masked_scores = scores.masked_fill(
        ~mask,
        float("-inf"),
    )

    attention_weights = torch.softmax(
        masked_scores,
        dim=-1,
    )

    output = (
        attention_weights
        @ V
    )

    return (
        output,
        attention_weights,
    )

## 42. Apply the Function to Our Q, K, and V

We will reuse the same:

```text
Q
K
V
```

from the earlier experiment.

The only difference is that attention is now both:

```text
scaled
```

and:

```text
causal
```

So token `0` will no longer be allowed to attend to token `1` or token `2`.

Token `1` will not be allowed to attend to token `2`.

Token `2` may attend to all three positions because it is the final position in this three-token sequence.

In [25]:
causal_output, causal_weights = (
    causal_scaled_dot_product_attention(
        Q,
        K,
        V,
    )
)

print("Causal scaled attention weights:")
print(causal_weights)

print("\nRow sums:")
print(
    causal_weights.sum(
        dim=-1
    )
)

print("\nCausal attention output:")
print(causal_output)

print("\nOutput shape:")
print(causal_output.shape)

Causal scaled attention weights:
tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00],
        [9.7880e-04, 9.9902e-01, 0.0000e+00],
        [2.7122e-02, 4.8644e-01, 4.8644e-01]])

Row sums:
tensor([1., 1., 1.])

Causal attention output:
tensor([[2.0000e+00, 1.0000e+00],
        [1.9576e-03, 3.9971e+00],
        [1.0271e+00, 3.4322e+00]])

Output shape:
torch.Size([3, 2])


## 43. Compare Non-Causal and Causal Attention

Earlier, scaled attention allowed every position to inspect every other position.

For token `0`, the scaled attention distribution was approximately spread across:

```text
token 0
token 1
token 2
```

With causal masking, token `0` is allowed to see only:

```text
token 0
```

So its attention weights must become:

```text
[1, 0, 0]
```

Token `1` may use:

```text
token 0
token 1
```

but token `2` must receive zero probability.

Token `2` may still use all three positions.

This means the attention matrix now contains both:

```text
learned / score-based preferences
```

and:

```text
hard structural constraints
```

The score determines how attention is distributed among allowed positions.

The mask determines which positions are forbidden entirely.

## 44. One More Important Distinction: Self-Attention

We have been calling this mechanism **self-attention**.

Why?

Because:

```text
Q
K
V
```

are all derived from the same sequence representation:

```text
X
```

Specifically:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

The sequence is therefore attending to itself.

This differs from **cross-attention**, where queries may come from one sequence while keys and values come from another.

For example:

```text
Q ← decoder representations

K, V ← encoder representations
```

That distinction becomes relevant in encoder-decoder architectures.

For our decoder-only GPT-style model, the core mechanism is:

```text
causal self-attention
```

## 45. Interpret the Actual Causal Attention Result

Our causal scaled attention weights were:

```text
[
    [1.0000, 0.0000, 0.0000],
    [0.0010, 0.9990, 0.0000],
    [0.0271, 0.4864, 0.4864],
]
```

This combines two different mechanisms:

```text
query-key scores
→ learned or data-dependent preferences

causal mask
→ hard structural constraint
```

### Token 0

Before causal masking, token `0` preferred token `2`.

But token `2` lies in its future.

The causal mask therefore changes:

```text
possible positions:
0, 1, 2
```

into:

```text
allowed positions:
0
```

so the final attention distribution must be:

```text
[1, 0, 0]
```

Its output is therefore exactly its own value vector:

```text
V[0] = [2, 1]
```

and we observe:

```text
output[0] = [2, 1]
```

---

### Token 1

Token `1` may attend to:

```text
position 0
position 1
```

but not position `2`.

Its causal attention weights are approximately:

```text
[0.0010, 0.9990, 0]
```

so its output is almost entirely the value vector from token `1`:

```text
output[1] ≈ [0.0020, 3.9971]
```

---

### Token 2

Token `2` is the final position in this sequence.

There are no future positions to hide, so it may attend to:

```text
0, 1, 2
```

Its distribution is:

```text
[0.0271, 0.4864, 0.4864]
```

and its output is a weighted combination of all three value vectors.

---

## The Important Separation

Attention now contains two conceptually different components:

```text
model preference
+
architectural constraint
```

The query-key scores answer:

> Among the positions I am allowed to inspect, which ones appear relevant?

The causal mask answers:

> Which positions am I allowed to inspect at all?

That distinction will remain true inside GPT.

# From Hand-Written Matrices to Learnable Attention

## 46. Our Projection Matrices Should Be Parameters

So far we manually created:

```text
W_Q
W_K
W_V
```

as fixed tensors.

But in an actual Transformer, these matrices are **learnable parameters**.

Training should determine how tokens should be projected into:

```text
query space
key space
value space
```

PyTorch's:

```python
nn.Linear(...)
```

is a convenient way to represent these learned projections.

For an input:

```text
X: [T, C]
```

a projection from:

```text
C → d_k
```

can be written as:

```python
nn.Linear(C, d_k, bias=False)
```

and produces:

```text
[T, C]
→
[T, d_k]
```

We use `bias=False` here to keep the operation close to the mathematical form we have been studying.

In [26]:
import torch.nn as nn

query_projection = nn.Linear(
    4,
    d_k,
    bias=False,
)

key_projection = nn.Linear(
    4,
    d_k,
    bias=False,
)

value_projection = nn.Linear(
    4,
    d_v,
    bias=False,
)

print(
    "Query weight shape:",
    query_projection.weight.shape,
)

print(
    "Key weight shape:",
    key_projection.weight.shape,
)

print(
    "Value weight shape:",
    value_projection.weight.shape,
)

Query weight shape: torch.Size([3, 4])
Key weight shape: torch.Size([3, 4])
Value weight shape: torch.Size([2, 4])


## 47. A PyTorch Shape Detail: `nn.Linear` Stores Its Weight Transposed

Earlier we wrote:

$$
Q = XW_Q
$$

with:

```text
X   : [T, C]
W_Q : [C, d_k]
```

But PyTorch stores an `nn.Linear` weight using the shape:

```text
[out_features, in_features]
```

So:

```python
nn.Linear(C, d_k)
```

contains:

```text
weight shape = [d_k, C]
```

and internally computes:

$$
XW^T + b
$$

Therefore, to reproduce our earlier matrix:

```text
W_Q: [C, d_k]
```

inside `nn.Linear`, we would copy:

```python
W_Q.T
```

into:

```python
query_projection.weight
```

This is a common PyTorch shape detail worth remembering:

```text
mathematical matrix we wrote: [in, out]

nn.Linear.weight:              [out, in]
```

In [27]:
with torch.no_grad():
    query_projection.weight.copy_(
        W_Q.T
    )

    key_projection.weight.copy_(
        W_K.T
    )

    value_projection.weight.copy_(
        W_V.T
    )

Q_linear = query_projection(x)
K_linear = key_projection(x)
V_linear = value_projection(x)

print("Manual Q:")
print(Q)

print("\nLinear Q:")
print(Q_linear)

print(
    "\nMax Q difference:",
    (Q - Q_linear).abs().max().item(),
)

print(
    "\nMax K difference:",
    (K - K_linear).abs().max().item(),
)

print(
    "\nMax V difference:",
    (V - V_linear).abs().max().item(),
)

Manual Q:
tensor([[1., 0., 1.],
        [2., 4., 0.],
        [2., 2., 1.]])

Linear Q:
tensor([[1., 0., 1.],
        [2., 4., 0.],
        [2., 2., 1.]], grad_fn=<MmBackward0>)

Max Q difference: 0.0

Max K difference: 0.0

Max V difference: 0.0


## 48. The Mathematics Did Not Change

Replacing our tensors with `nn.Linear` did not introduce a new attention algorithm.

We still compute:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

The difference is that PyTorch now registers the projection matrices as model parameters.

That means they can participate in:

```text
forward pass
↓
loss
↓
backpropagation
↓
gradients
↓
optimizer update
```

So during language-model training, the model can learn:

```text
what queries should represent
what keys should represent
what values should represent
```

from the next-token objective.

This follows the same principle we saw for embeddings in Module 3:

> We do not manually assign semantic roles to the learned vectors. The training objective shapes them through gradient descent.

## 49. Package the Mechanism into a PyTorch Module

We now have enough pieces to package single-head causal self-attention into an `nn.Module`.

The module will contain three learnable projections:

```text
X → Q
X → K
X → V
```

and internally perform:

```text
Q @ K.T
↓
divide by sqrt(d_k)
↓
causal mask
↓
softmax
↓
attention weights @ V
```

This is the first point where our implementation starts looking structurally similar to a real Transformer implementation.

We are still keeping it deliberately simple:

```text
one sequence
one attention head
no batching
no dropout
no output projection
```

Those pieces will be introduced only after the single-head mechanism is completely clear.

In [28]:
class SingleHeadCausalSelfAttention(
    nn.Module
):
    def __init__(
        self,
        input_dim,
        key_dim,
        value_dim,
    ):
        super().__init__()

        self.key_dim = key_dim

        self.query = nn.Linear(
            input_dim,
            key_dim,
            bias=False,
        )

        self.key = nn.Linear(
            input_dim,
            key_dim,
            bias=False,
        )

        self.value = nn.Linear(
            input_dim,
            value_dim,
            bias=False,
        )

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = (
            Q @ K.T
            / (self.key_dim ** 0.5)
        )

        T = x.shape[0]

        causal_mask = torch.tril(
            torch.ones(
                T,
                T,
                dtype=torch.bool,
                device=x.device,
            )
        )

        scores = scores.masked_fill(
            ~causal_mask,
            float("-inf"),
        )

        attention_weights = torch.softmax(
            scores,
            dim=-1,
        )

        output = (
            attention_weights
            @ V
        )

        return (
            output,
            attention_weights,
        )

## 50. Inspect the Learnable Parameters

Before running the module, notice what has changed.

Earlier we manually supplied:

```text
W_Q
W_K
W_V
```

Now the module owns these parameters.

PyTorch will expose them through:

```python
model.parameters()
```

and:

```python
model.named_parameters()
```

This means an optimizer can update them automatically.

Let's inspect their names and shapes.

In [29]:
torch.manual_seed(42)

attention = (
    SingleHeadCausalSelfAttention(
        input_dim=4,
        key_dim=3,
        value_dim=2,
    )
)

for name, parameter in (
    attention.named_parameters()
):
    print(
        f"{name:15s}",
        tuple(parameter.shape),
    )

query.weight    (3, 4)
key.weight      (3, 4)
value.weight    (2, 4)


In [30]:
with torch.no_grad():
    learned_output, learned_weights = (
        attention(x)
    )

print("Attention weights:")
print(learned_weights)

print("\nRow sums:")
print(
    learned_weights.sum(
        dim=-1
    )
)

print("\nAttention output:")
print(learned_output)

print("\nOutput shape:")
print(learned_output.shape)

Attention weights:
tensor([[1.0000, 0.0000, 0.0000],
        [0.6936, 0.3064, 0.0000],
        [0.3283, 0.3096, 0.3621]])

Row sums:
tensor([1., 1., 1.])

Attention output:
tensor([[-0.5359,  0.4987],
        [-0.6972, -0.2170],
        [-0.8912, -0.5570]])

Output shape:
torch.Size([3, 2])


## 51. Why These Numbers Differ from Our Earlier Example

The attention weights from this module will probably not match our earlier manually constructed Q/K/V example.

That is expected.

Previously, we explicitly chose:

```text
W_Q
W_K
W_V
```

Now `nn.Linear` initialized its projection weights randomly.

So:

```text
different projection matrices
↓
different Q, K, V
↓
different query-key scores
↓
different attention weights
↓
different outputs
```

The architecture is the same.

The parameters are different.

During actual model training, gradient descent will modify these parameters based on next-token prediction loss.

That is how useful attention patterns can emerge.

## 52. One Remaining Architectural Issue

Our input has shape:

```text
[T, C]
=
[3, 4]
```

but our attention output has shape:

```text
[T, d_v]
=
[3, 2]
```

That was useful pedagogically because it showed that the value dimension is independent.

But a Transformer block eventually uses a residual connection of the form:

$$
X + \text{Attention}(X)
$$

Addition requires matching shapes.

We cannot directly add:

```text
[T, 4]
+
[T, 2]
```

So practical Transformer attention introduces another learned projection after attention.

Conceptually:

```text
attention output
[T, d_v]

↓

output projection

[T, C]
```

This projection is commonly called:

$$
W_O
$$

or the **output projection**.

Multi-head attention makes this even more important because several attention heads will be concatenated and then projected back into the model dimension.

We will introduce that in the next notebook.

# Notebook 1 Summary

## 53. What We Built

We started without assuming the Transformer attention formula.

Instead, we built the mechanism piece by piece.

### 1. Pairwise token interaction

Starting from:

```text
X: [T, C]
```

we first calculated:

$$
XX^T
$$

to produce one pairwise score for every pair of token positions.

---

### 2. Attention weights

We normalized those scores with softmax:

$$
A
=
\text{softmax}(XX^T)
$$

and used the resulting weights to combine token representations.

This established the core pattern:

```text
compare
→ normalize
→ aggregate
```

---

### 3. Queries, keys, and values

We separated the three roles using:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

Queries and keys determine:

```text
where to attend
```

while values determine:

```text
what information is retrieved
```

---

### 4. Scaled dot-product attention

We experimentally observed:

$$
\operatorname{Std}(q \cdot k)
\approx
\sqrt{d_k}
$$

As `d_k` increased, unscaled attention became increasingly sharp.

Dividing by:

$$
\sqrt{d_k}
$$

kept score scale and attention sharpness approximately stable.

This produced:

$$
\text{Attention}(Q,K,V)
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

---

### 5. Causal masking

For autoregressive language modeling, future tokens must not influence earlier positions.

We therefore masked:

$$
j > i
$$

with:

$$
-\infty
$$

before softmax.

This guarantees:

```text
future attention probability = 0
```

while allowed positions still sum to `1`.

---

### 6. Learnable projections

Finally, we replaced manually specified projection matrices with:

```python
nn.Linear
```

so that:

```text
W_Q
W_K
W_V
```

can be learned through backpropagation.

---

## The Core Decoder Attention Pipeline

We now understand:

```text
X
│
├── W_Q → Q
├── W_K → K
└── W_V → V

Q @ K.T
↓
scale by 1 / sqrt(d_k)
↓
causal mask
↓
softmax
↓
attention weights
↓
weighted sum of V
↓
context-aware token representations
```

The essential equation is:

$$
\boxed{
\text{CausalSelfAttention}(Q,K,V)
=
\text{softmax}
\left(
\text{causal-mask}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
\right)V
}
$$

---

## What Comes Next

So far we have used only one attention head.

A single head produces one learned way of comparing tokens.

A Transformer instead uses multiple attention heads in parallel.

Each head gets its own learned projections and can potentially learn a different relationship between tokens.

The next notebook will build:

```text
multiple attention heads
↓
concatenate their outputs
↓
output projection
↓
residual-compatible representation
```

and will also extend the implementation from:

```text
[T, C]
```

to the batched form used during real training:

```text
[B, T, C]
```

That brings us to **multi-head causal self-attention**.

## 54. Experimental Takeaways

The experiments in this notebook produced several useful sanity checks.

### Dot-product scaling

For random standard-normal queries and keys:

```text
d_k =    4 → dot-product std ≈  2.04
d_k =   16 → dot-product std ≈  4.02
d_k =   64 → dot-product std ≈  7.97
d_k =  256 → dot-product std ≈ 16.01
d_k = 1024 → dot-product std ≈ 31.72
```

matching the expected relationship:

$$
\operatorname{Std}(q \cdot k)
\approx
\sqrt{d_k}
$$

After dividing by $\sqrt{d_k}$, the standard deviation stayed close to `1`.

### Attention sharpness

Without scaling, increasing `d_k` pushed random attention toward nearly one-hot distributions:

```text
d_k =    4 → average max weight ≈ 0.43
d_k = 1024 → average max weight ≈ 0.96
```

With scaling, the average maximum remained near:

```text
0.24 – 0.25
```

across dimensions.

### Causal masking

Future-token attention probabilities became exactly:

```text
0
```

while each allowed attention row continued to sum to:

```text
1
```

### PyTorch projection sanity check

Replacing explicit matrix multiplication with `nn.Linear` reproduced the manually constructed projections exactly:

```text
max Q difference = 0
max K difference = 0
max V difference = 0
```

These checks connect the Transformer attention formula to observable numerical behavior rather than treating it as a formula to memorize.